# Analyse Exploratoire - JO 2028 (Sujet 3)

Ce notebook retrace la demarche d'analyse et les principaux insights pour le projet JO 2028.

## Objectifs

- Explorer le dataset olympique et produire des indicateurs cles.
- Comparer la domination historique par pays et par sport.
- Mettre en avant l'evolution de la participation femme/homme.
- Proposer un focus sport (Athletics / Swimming).
- Construire une baseline simple pour les cibles 2028.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "olympics_dataset.csv"

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
})

In [ ]:
def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


def prepare_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["Name", "Team", "NOC", "Sport", "Event", "City", "Medal", "Season", "Sex"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    df["has_medal"] = df["Medal"].ne("No medal") & df["Medal"].notna()
    return df


def filter_season(df: pd.DataFrame, season: str | None) -> pd.DataFrame:
    if not season:
        return df
    season_norm = season.strip().lower()
    return df[df["Season"].astype(str).str.strip().str.lower() == season_norm]


def get_recent_years(df: pd.DataFrame, recent_n: int) -> list[int]:
    years = sorted(df["Year"].dropna().unique().tolist())
    years = [int(y) for y in years]
    if recent_n <= 0:
        return years
    return years[-recent_n:]


raw_df = load_dataset(DATA_PATH)
raw_df.head(3)

## Preparation et filtrage (Summer)

On se place sur les editions d'ete pour rester coherent avec JO 2028.

In [ ]:
df = prepare_dataset(raw_df)
df = filter_season(df, "Summer")

df.shape

## KPIs de base

In [ ]:
kpis = {
    "lignes": len(df),
    "athletes_uniques": df["player_id"].nunique(),
    "editions": df["Year"].nunique(),
    "sports": df["Sport"].nunique(),
    "epreuves": df["Event"].nunique(),
}

pd.DataFrame([kpis])

## Domination historique: top pays et top sports

In [ ]:
medals = df[df["has_medal"]]

top_countries = (
    medals.groupby("NOC", as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(15)
)

top_sports = (
    medals.groupby("Sport", as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(15)
)

display(top_countries)
display(top_sports)

ax = top_countries.set_index("NOC")["size"].sort_values().plot.barh()
ax.set_title("Top 15 pays par medailles (historique)")
ax.set_xlabel("Medailles")
plt.show()

ax = top_sports.set_index("Sport")["size"].sort_values().plot.barh()
ax.set_title("Top 15 sports par medailles (historique)")
ax.set_xlabel("Medailles")
plt.show()

## Editions recentes (comparaison)

In [ ]:
recent_years = get_recent_years(df, 5)
medals_recent = medals[medals["Year"].isin(recent_years)]

top_recent = (
    medals_recent.groupby("NOC", as_index=False)
    .size()
    .sort_values("size", ascending=False)
    .head(15)
)

top_recent

In [ ]:
ax = top_recent.set_index("NOC")["size"].sort_values().plot.barh()
ax.set_title("Top 15 pays (editions recentes)")
ax.set_xlabel("Medailles")
plt.show()

## Medailles par annee

In [ ]:
medals_by_year = (
    medals.groupby(["Year", "Medal"], as_index=False)
    .size()
    .pivot(index="Year", columns="Medal", values="size")
    .fillna(0)
    .reset_index()
)

medal_cols = [c for c in medals_by_year.columns if c != "Year"]
ax = medals_by_year.set_index("Year")[medal_cols].plot()
ax.set_title("Medailles par annee et type")
ax.set_xlabel("Annee")
ax.set_ylabel("Medailles")
plt.show()

## Participation femme/homme

In [ ]:
participation = (
    df.groupby(["Year", "Sex"], as_index=False)
    .size()
    .pivot(index="Year", columns="Sex", values="size")
    .fillna(0)
    .reset_index()
)
for col in ["F", "M"]:
    if col not in participation.columns:
        participation[col] = 0
participation["total"] = participation["F"] + participation["M"]
participation["female_share"] = participation["F"] / participation["total"].replace(0, pd.NA)

ax = participation.set_index("Year")[["F", "M"]].plot()
ax.set_title("Participation par sexe")
ax.set_xlabel("Annee")
ax.set_ylabel("Participations")
plt.show()

ax = participation.set_index("Year")[["female_share"]].plot()
ax.set_title("Part des femmes")
ax.set_xlabel("Annee")
ax.set_ylabel("Part")
plt.show()

## Focus sport (Athletics / Swimming)

In [ ]:
def focus_sport_summary(medals_df: pd.DataFrame, sport: str, top_n: int = 10) -> None:
    focus = medals_df[medals_df["Sport"].astype(str).str.strip().str.lower() == sport.lower()]
    top_c = (
        focus.groupby("NOC", as_index=False)
        .size()
        .sort_values("size", ascending=False)
        .head(top_n)
    )
    by_year = focus.groupby("Year", as_index=False).size().sort_values("Year")

    display(top_c)

    ax = top_c.set_index("NOC")["size"].sort_values().plot.barh()
    ax.set_title(f"Top pays - {sport}")
    ax.set_xlabel("Medailles")
    plt.show()

    ax = by_year.set_index("Year")["size"].plot()
    ax.set_title(f"Medailles {sport} par annee")
    ax.set_xlabel("Annee")
    ax.set_ylabel("Medailles")
    plt.show()


focus_sport_summary(medals, "Athletics", top_n=10)
focus_sport_summary(medals, "Swimming", top_n=10)

## Baseline prediction 2028 (moyenne sur les 3 dernieres editions)

In [ ]:
pred_years = recent_years[-3:] if len(recent_years) >= 3 else recent_years
pred_medals = medals[medals["Year"].isin(pred_years)]

prediction = (
    pred_medals.groupby("NOC", as_index=False)
    .size()
    .assign(recent_years=", ".join(str(y) for y in pred_years))
)
prediction["avg_per_edition"] = prediction["size"] / max(len(pred_years), 1)

prediction.sort_values("avg_per_edition", ascending=False).head(10)